# Module 22 – Titanic Survival Prediction using Naive Bayes

Comprehensive EDA, preprocessing, visualization, model evaluation, interpretation and cross-validation.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

In [ ]:
df=pd.read_csv('tested-selected-columns.csv')
print(df.head())
print(df.info())
print(df.describe(include='all'))
print('\nMissing Values:\n',df.isnull().sum())

## Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(6,4))
sns.heatmap(df.isnull(),cbar=False)
plt.title('Missing Value Analysis')
plt.show()

df['Survived'].value_counts().plot(kind='bar',title='Survival Distribution')
plt.show()

sns.histplot(df['Age'],kde=True)
plt.title('Age Distribution')
plt.show()

sns.barplot(data=df,x='Sex',y='Survived')
plt.title('Survival Rate by Gender')
plt.show()

sns.barplot(data=df,x='Pclass',y='Survived')
plt.title('Survival Rate by Passenger Class')
plt.show()

## Preprocessing Rationale
Age and Fare contain missing values, therefore median imputation is used because it is robust to outliers. Sex, Name and Ticket are categorical variables; OneHotEncoder is used instead of LabelEncoder because these features are nominal and have no natural ordering.

In [ ]:
X=df.drop('Survived',axis=1)
y=df['Survived']

num_features=X.select_dtypes(include=['int64','float64']).columns
cat_features=X.select_dtypes(include=['object']).columns

preprocessor=ColumnTransformer([
('num',Pipeline([('imputer',SimpleImputer(strategy='median'))]),num_features),
('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                 ('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_features)
])

X_processed=preprocessor.fit_transform(X).toarray()
X_train,X_test,y_train,y_test=train_test_split(X_processed,y,test_size=0.2,random_state=42)

In [ ]:
model=GaussianNB()
model.fit(X_train,y_train)
y_pred=model.predict(X_test)

print('Accuracy:',accuracy_score(y_test,y_pred))
print('\nClassification Report\n',classification_report(y_test,y_pred))

In [ ]:
cm=confusion_matrix(y_test,y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

tn,fp,fn,tp=cm.ravel()
print('True Positive Rate:',tp/(tp+fn))
print('False Negative Rate:',fn/(tp+fn))

In [ ]:
scores=cross_val_score(GaussianNB(),X_processed,y,cv=5,scoring='accuracy')
print('Cross Validation Scores:',scores)
print('Mean CV Accuracy:',scores.mean())

## Findings and Insights

1. Female passengers show a much higher survival rate than male passengers. Passenger class also influences survival, with higher classes generally performing better.

2. Missing values are concentrated mainly in the Age column. Median imputation is preferred because it reduces the influence of extreme values. OneHotEncoding avoids introducing artificial ordinal relationships among categorical features.

3. The confusion matrix reveals how many passengers were correctly and incorrectly classified. The true positive rate indicates how effectively the model identifies survivors, while the false negative rate shows survivors incorrectly predicted as non-survivors.

4. Cross-validation provides a robustness assessment by testing the model on multiple folds. Consistent scores indicate stable performance.

## Limitations and Improvements

- Gaussian Naive Bayes assumes feature independence, which is often unrealistic in passenger data.
- Additional feature engineering (family size, title extraction from names, ticket groups) may improve performance.
- Other Naive Bayes variants and ensemble methods could be compared for better accuracy.
- Hyperparameter tuning and feature selection may further improve results.
